# Database Setup: Coffee Reviews Schema

Splits the cleaned CSV into two SQLite tables:
- **coffee_reviews**: structured metadata (name, roaster, location, scores, etc.)
- **coffee_review_text**: review text used for embedding/retrieval

Each row in both tables is linked by a shared `review_uid`.

In [1]:
from pathlib import Path

# Project paths
cwd = Path.cwd()
PROJECT_ROOT = cwd.parents[0]
DATA_ROOT = PROJECT_ROOT / "data"
SQL_DATA_ROOT = PROJECT_ROOT / "db"

SQL_DATA_ROOT.mkdir(exist_ok=True, parents=True)

In [2]:
# imports and Configuration
import uuid
import sqlite3
import pandas as pd

In [3]:
# provide file paths
CSV_PATH = DATA_ROOT / "processed" / "01_coffee_reviews_clean.csv"
SQL_DB_PATH = SQL_DATA_ROOT / "coffee_reviews.db"

## Step 1: Load and Inspect Cleaned CSV

Load the cleaned CSV into a DataFrame and confirm the structure matches
what we expect before doing any transformations.

In [4]:
df = pd.read_csv(CSV_PATH)

print(f"Shape: {df.shape}")
df.info()
df.head()

Shape: (2280, 18)
<class 'pandas.DataFrame'>
RangeIndex: 2280 entries, 0 to 2279
Data columns (total 18 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   slug          2280 non-null   str    
 1   rating        2280 non-null   int64  
 2   roaster       2280 non-null   str    
 3   name          2280 non-null   str    
 4   location      2280 non-null   str    
 5   origin        2280 non-null   str    
 6   roast         2280 non-null   str    
 7   est_price     2280 non-null   str    
 8   review_date   2280 non-null   str    
 9   agtron        2280 non-null   str    
 10  aroma         2253 non-null   float64
 11  acid          1945 non-null   float64
 12  body          2277 non-null   float64
 13  flavor        2277 non-null   float64
 14  aftertaste    2277 non-null   float64
 15  desc_1        2280 non-null   str    
 16  desc_3        2280 non-null   str    
 17  desc_2_clean  2279 non-null   str    
dtypes: float64(5), int64(

,slug,rating,roaster,name,location,origin,roast,est_price,review_date,agtron,aroma,acid,body,flavor,aftertaste,desc_1,desc_3,desc_2_clean
0,https://www.coffeereview.com/review/sweety-esp...,95,A.R.C.,“Sweety” Espresso Blend,"Hong Kong, China",Panama; Ethiopia,Medium-Light,HKD $250/227 grams,2017-11-01,50/73,9.0,NaN,9.0,9.0,9.0,"Evaluated as espresso. Sweet-toned, deeply ric...",A radiant espresso blend that shines equally i...,An espresso blend comprised of coffees from Pa...
1,https://www.coffeereview.com/review/flora-blen...,94,A.R.C.,Flora Blend Espresso,"Hong Kong, China",Africa; Asia Pacific,Medium-Light,HKD $158/227 grams,2017-11-01,54/77,9.0,NaN,9.0,9.0,8.0,"Evaluated as espresso. Sweetly tart, floral-to...","A floral-driven straight shot, amplified with ...",An espresso blend comprised of coffees from Af...
2,https://www.coffeereview.com/review/ethiopia-s...,92,Revel Coffee,Ethiopia Shakiso Mormora,"Billings, Montana","Guji Zone, southern Ethiopia",Medium-Light,$16.00/12 ounces,2017-11-01,54/70,9.0,8.0,8.0,9.0,8.0,"Crisply sweet, cocoa-toned. Lemon blossom, roa...","A gently spice-toned, floral- driven wet-proce...",This coffee tied for the third-highest rating ...
3,https://www.coffeereview.com/review/ethiopia-s...,92,Roast House,Ethiopia Suke Quto,"Spokane, Washington","Guji Zone, Oromia Region, south-central Ethiopia",Medium-Light,$19.00/16 ounces,2017-11-01,53/79,8.0,8.0,9.0,9.0,8.0,"Delicate, sweetly spice-toned. Pink peppercorn...",Lavender-like flowers and hints of zesty pink ...,This coffee tied for the third-highest rating ...
4,https://www.coffeereview.com/review/ethiopia-g...,94,Big Creek Coffee Roasters,Ethiopia Gedeb Halo Beriti,"Hamilton, Montana","Gedeb District, Gedeo Zone, southern Ethiopia",Medium,$16.50/12 ounces,2017-11-01,48/70,9.0,9.0,9.0,9.0,8.0,"Deeply sweet, subtly pungent. Honey, pear, tan...",A deeply and generously lush cup saved from se...,Southern Ethiopia coffees like this one are pr...


## Step 2: Fix Data Types

`review_date` is currently stored as text format. Convert it to a proper date type so it can
be sorted, filtered, and stored correctly in SQL.

In [5]:
df['review_date'] = pd.to_datetime(df['review_date'])

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2280 entries, 0 to 2279
Data columns (total 18 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   slug          2280 non-null   str           
 1   rating        2280 non-null   int64         
 2   roaster       2280 non-null   str           
 3   name          2280 non-null   str           
 4   location      2280 non-null   str           
 5   origin        2280 non-null   str           
 6   roast         2280 non-null   str           
 7   est_price     2280 non-null   str           
 8   review_date   2280 non-null   datetime64[us]
 9   agtron        2280 non-null   str           
 10  aroma         2253 non-null   float64       
 11  acid          1945 non-null   float64       
 12  body          2277 non-null   float64       
 13  flavor        2277 non-null   float64       
 14  aftertaste    2277 non-null   float64       
 15  desc_1        2280 non-null   str           
 16 

## Step 3: Generate UIDs and Split into Two Tables

Create a stable `review_uid` for every row, then split the DataFrame
into structured metadata (`coffee_reviews`) and narrative text
(`coffee_review_text`), linked by that UID.

In [7]:
df['review_uid'] = [str(uuid.uuid4()) for _ in range(len(df))]

In [8]:
df.columns

Index(['slug', 'rating', 'roaster', 'name', 'location', 'origin', 'roast',
       'est_price', 'review_date', 'agtron', 'aroma', 'acid', 'body', 'flavor',
       'aftertaste', 'desc_1', 'desc_3', 'desc_2_clean', 'review_uid'],
      dtype='str')

In [9]:
# Table 1: structured metadata
metadata_cols = [
    'review_uid', 'slug', 'rating', 'roaster', 'name', 'location',
    'origin', 'roast', 'est_price', 'review_date', 'agtron',
    'aroma', 'acid', 'body', 'flavor', 'aftertaste'
]

coffee_reviews_df = df[metadata_cols].copy()

In [20]:
# Table 2: Reviews Texts
text_cols = ['review_uid', 'desc_1', 'desc_3', 'desc_2_clean']
coffee_reviews_text_df = df[text_cols].copy()

In [21]:
coffee_reviews_text_df.head()

,review_uid,desc_1,desc_3,desc_2_clean
0,541f70da-4449-4acd-865d-38cbca91a597,"Evaluated as espresso. Sweet-toned, deeply ric...",A radiant espresso blend that shines equally i...,An espresso blend comprised of coffees from Pa...
1,48c30ff5-4757-4f70-823a-66578a7d53a5,"Evaluated as espresso. Sweetly tart, floral-to...","A floral-driven straight shot, amplified with ...",An espresso blend comprised of coffees from Af...
2,ab98d11a-575b-44e6-9826-0b841a1d5e11,"Crisply sweet, cocoa-toned. Lemon blossom, roa...","A gently spice-toned, floral- driven wet-proce...",This coffee tied for the third-highest rating ...
3,01442a89-853a-44db-b9d2-d7a239c7bc45,"Delicate, sweetly spice-toned. Pink peppercorn...",Lavender-like flowers and hints of zesty pink ...,This coffee tied for the third-highest rating ...
4,a22d7940-0505-43ac-95d4-79efd6e024df,"Deeply sweet, subtly pungent. Honey, pear, tan...",A deeply and generously lush cup saved from se...,Southern Ethiopia coffees like this one are pr...


## Step 4: Create SQLite Tables

Define the schema for both tables and create them in the database.
`review_uid` is the primary key of `coffee_reviews`, and also serves
as the primary key of `coffee_review_text` (a true 1:1 relationship,
enforced by the foreign key back to `coffee_reviews`).

In [25]:
# schema creation
conn = sqlite3.connect(SQL_DB_PATH)
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS coffee_reviews_text")
cursor.execute("DROP TABLE IF EXISTS coffee_reviews")

sql_statements = [
    """ CREATE TABLE coffee_reviews (
        review_uid TEXT PRIMARY KEY,
        slug TEXT,
        rating INTEGER,
        roaster TEXT,
        name TEXT,
        location TEXT,
        origin TEXT,
        roast TEXT,
        est_price TEXT,
        review_date DATE,
        agtron TEXT,
        aroma REAL,
        acid REAL,
        body REAL,
        flavor REAL,
        aftertaste REAL 
        );""" ,

    """ CREATE TABLE coffee_reviews_text (
        review_uid TEXT PRIMARY KEY,
        desc_1 TEXT,
        desc_2_clean TEXT,
        desc_3 TEXT,
        FOREIGN KEY (review_uid) REFERENCES coffee_reviews (review_uid)
    );"""   
]

cursor.execute(sql_statements[0])

cursor.execute(sql_statements[1])

conn.commit()
print("Tables created successfully")

Tables created successfully


## Step 5: Insert Data

Write both DataFrames into their tables. We use `if_exists='append'`
here rather than `'replace'`, since the tables (with their primary
keys and foreign key constraint) were already created in the cell
above - `'replace'` would drop them and reinsert with no schema.

In [26]:
coffee_reviews_df.to_sql('coffee_reviews', conn, if_exists='append', index=False)
coffee_reviews_text_df.to_sql('coffee_reviews_text', conn, if_exists='append', index=False)
conn.commit()

print(f'Inserted {len(coffee_reviews_df)} rows into coffee_reviews')
print(f'Inserted {len(coffee_reviews_text_df)} rows into coffee_reviews_text')

Inserted 2280 rows into coffee_reviews
Inserted 2280 rows into coffee_review_text


In [28]:
# Row counts
review_count = cursor.execute('SELECT COUNT(*) FROM coffee_reviews').fetchone()[0]
text_count = cursor.execute('SELECT COUNT(*) FROM coffee_reviews_text').fetchone()[0]
print(f'coffee_reviews: {review_count} rows')
print(f'coffee_reviews_text: {text_count} rows')

assert review_count == len(df), 'Row count mismatch in coffee_reviews!'
assert text_count == len(df), 'Row count mismatch in coffee_review_text!'
print('Row counts match source data.')

# Sample join, reading review_date back as a real date (SQLite stores it as text internally)
sample = pd.read_sql_query("""
    SELECT cr.review_uid, cr.roaster, cr.rating, cr.review_date, crt.desc_2_clean
    FROM coffee_reviews cr
    JOIN coffee_reviews_text crt ON cr.review_uid = crt.review_uid
    LIMIT 3
""", conn, parse_dates=['review_date'])

print(sample)
print(sample.dtypes)

conn.close()

coffee_reviews: 2280 rows
coffee_reviews_text: 2280 rows
Row counts match source data.
                             review_uid       roaster  rating review_date  \
0  541f70da-4449-4acd-865d-38cbca91a597        A.R.C.      95  2017-11-01   
1  48c30ff5-4757-4f70-823a-66578a7d53a5        A.R.C.      94  2017-11-01   
2  ab98d11a-575b-44e6-9826-0b841a1d5e11  Revel Coffee      92  2017-11-01   

                                        desc_2_clean  
0  An espresso blend comprised of coffees from Pa...  
1  An espresso blend comprised of coffees from Af...  
2  This coffee tied for the third-highest rating ...  
review_uid                 str
roaster                    str
rating                   int64
review_date     datetime64[us]
desc_2_clean               str
dtype: object
